# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [21]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [22]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: c:\Users\osaci\Desktop\Proiect_Inginerie_AI\echochamber-project-team-1
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [23]:
student_id = "student_02"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [24]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [25]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [26]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(20)# completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
VeridicaRO                           91
TVRcanaluloficial                    85
hotnews_ro                           30
Canal33Romania                       25
realitatea2025                       23
Name: count, dtype: int64

In [27]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [28]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [29]:
# IMPORTANT - SCHIMBA PROMTUL DE MAI JOS PENTRU A SE POTRIVI CU CERINȚELE TALE ȘI ASIGURĂ-TE CĂ RESPECTĂ STRUCTURA SOLICITATĂ
# include în prompt instrucțiuni clare pentru fiecare dintre cele 7 elemente pe care vrei să le extragi și asigură-te că modelul înțelege că trebuie să returneze un JSON valid cu exact acele chei
# inlocueste "..." cu instrucțiuni clare pentru fiecare element
# Prompt de sistem: definește rolul modelului
# poti pune si alte axe de analiza care te intereseaza


SYSTEM_PROMPT = SYSTEM_PROMPT = """
You are a political discourse analyst.

Your task is to analyze Romanian political YouTube comments in a structured, neutral, and consistent way.

Focus especially on anti-establishment discourse, emotional tone, distrust, cynicism, and skepticism toward political actors, institutions, media, elites, or official narratives.

Important rules:
- Return only valid JSON.
- Do not add explanations outside the JSON.
- Do not invent information that is not present in the comment.
- If something is unclear, write "unclear".
- Distinguish clearly between sentiment and stance.
- A comment can have negative sentiment but still support a target by attacking that target's opponents.
- Pay attention to sarcasm, irony, cynicism, vague references, multiple targets, and anti-system language.
- Do not judge whether the comment is factually or politically correct.
"""

USER_PROMPT_TEMPLATE =  """
Read the following Romanian political YouTube comment and identify exactly these 7 elements:

1. target:
The main person, party, institution, group, ideology, media actor, elite, or political object discussed in the comment.
If there are multiple targets, mention the main target and secondary targets.

2. stance:
The author's position toward the main target.
Use one of: supportive, opposed, neutral, mixed, unclear.

3. sentiment:
The general emotional tone of the comment.
Use one of: positive, negative, neutral, mixed, unclear.
Briefly indicate the dominant emotion if visible: anger, fear, contempt, hope, disappointment, distrust, frustration, or unclear.

4. topic:
The main political or social issue discussed.
Examples: corruption, elections, economy, nationalism, foreign policy, democracy, media, justice, political leadership, social conflict, institutional distrust.

5. anti_establishment_frame:
Does the comment express distrust, opposition, or skepticism toward the political system, parties, institutions, elites, media, politicians, or official narratives?
Use one of: yes, no, partially, unclear.
Briefly mention why.

6. cynicism:
Does the comment suggest cynical distrust, moral disgust, fatalism, or the belief that politicians/institutions are corrupt, fake, manipulative, or hopeless?
Use one of: high, medium, low, none, unclear.
Briefly mention the clue from the comment.

7. interpretation_problem:
Mention whether sarcasm, irony, ambiguity, vague references, missing context, slang, multiple targets, or emotional exaggeration make the comment difficult to interpret.
If there is no major problem, write "none".

Important:
Return only valid JSON with exactly these keys:

{{
  "target": "...",
  "stance": "...",
  "sentiment": "...",
  "topic": "...",
  "anti_establishment_frame": "...",
  "cynicism": "...",
  "interpretation_problem": "..."
}}

Comment:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [30]:
import os

# Curățăm variabile OpenAI care pot intra ca headers și pot cauza UnicodeEncodeError
for key in [
    "OPENAI_API_KEY",
    "OPENAI_ORG_ID",
    "OPENAI_PROJECT",
    "OPENAI_ORGANIZATION"
]:
    os.environ.pop(key, None)

print("OpenAI env headers cleared.")

OpenAI env headers cleared.


In [31]:
from openai import OpenAI

model = "gemini-2.5-flash-lite"
temperature = 0.2

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    default_headers={}
)

print("Client ready.")
print("Model:", model)
print("Gemini key ASCII:", GEMINI_API_KEY.isascii())

Client ready.
Model: gemini-2.5-flash-lite
Gemini key ASCII: True


In [34]:
import unicodedata
import re
import pandas as pd

def to_ascii_safe(text):
    """
    Transformă textul într-o variantă ASCII-safe.
    Elimină diacritice, emoji și caractere speciale care pot da erori la API.
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", errors="ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [35]:
def annotate_comment(comment_text):
    safe_comment = to_ascii_safe(comment_text)
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=safe_comment)

    safe_system_prompt = to_ascii_safe(SYSTEM_PROMPT)
    safe_user_prompt = to_ascii_safe(prompt)

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": safe_system_prompt},
            {"role": "user", "content": safe_user_prompt}
        ]
    )

    return response.choices[0].message.content

In [36]:
print(annotate_comment("Guvernul a anuntat noi masuri economice."))

```json
{
  "target": "Guvernul",
  "stance": "neutral",
  "sentiment": "neutral",
  "topic": "economy",
  "anti_establishment_frame": "no",
  "cynicism": "none",
  "interpretation_problem": "none"
}
```


## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [37]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"```json\n{\n ""target"": ""George Simion (main),..."
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,"```json\n{\n ""target"": ""Romanian government/m..."
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"```json\n{\n ""target"": ""Main target: AUR (pol..."
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","```json\n{\n ""target"": ""George Simion (main),..."
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"```json\n{\n ""target"": ""George Simion (main),..."
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...","```json\n{\n ""target"": ""George Simion"",\n ""s..."
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,"```json\n{\n ""target"": ""George (likely a poli..."


# 9. Verificam rezultatele

In [39]:
results_df.model_output[0]

'```json\n{\n  "target": "George Simion",\n  "stance": "supportive",\n  "sentiment": "positive",\n  "topic": "political leadership",\n  "anti_establishment_frame": "no",\n  "cynicism": "none",\n  "interpretation_problem": "none"\n}\n```'

In [40]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [41]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,George Simion,supportive,positive,,political leadership,none,
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"George Simion (main), Parliament (secondary)",supportive,positive,,"political leadership, parliamentary activity",none,
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,George Simion,supportive,positive,,political leadership,none,
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,Romanian government/military leadership,opposed,negative,,national security/military preparedness,none,
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,Main target: AUR (political party) and George ...,"Supportive towards AUR and George Simion, oppo...","Mixed. Positive towards AUR and its leaders, n...",,"Political leadership, government change, elect...",None,
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","George Simion (main), Klaus Iohannis, Fritz (s...",mixed,negative,,"political leadership, public spending, politic...",none,
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,George Simion,supportive,positive,,political leadership,none,
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"George Simion (main), Vlad Tepes (secondary, u...",mixed,neutral,,political leadership,none,
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...",George Simion,supportive,positive,,political leadership,none,
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,George (likely a politician or public figure),supportive,positive,,political leadership,none,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [42]:
# Salvăm rezultatele într-un CSV în folderul student_02

from pathlib import Path

results_output_dir = ROOT / "notebooks" / "student_02" / "outputs"
results_output_dir.mkdir(parents=True, exist_ok=True)

csv_output_file = results_output_dir / "student_02_tema1_prompt_results.csv"

results_df.to_csv(csv_output_file, index=False, encoding="utf-8-sig")

print("CSV saved to:", csv_output_file)
print("Number of analyzed comments:", len(results_df))

CSV saved to: c:\Users\osaci\Desktop\Proiect_Inginerie_AI\echochamber-project-team-1\notebooks\student_02\outputs\student_02_tema1_prompt_results.csv
Number of analyzed comments: 10


In [43]:
# Verificăm că fișierul CSV se poate deschide corect

check_df = pd.read_csv(csv_output_file)

print("Loaded CSV rows:", len(check_df))
print("Columns:", check_df.columns.tolist())

check_df.head()

Loaded CSV rows: 10
Columns: ['source_channel', 'video_title', 'comment_text', 'model_output']


,source_channel,video_title,comment_text,model_output
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"```json\n{\n ""target"": ""George Simion (main),..."
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,"```json\n{\n ""target"": ""Romanian government/m..."
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"```json\n{\n ""target"": ""Main target: AUR (pol..."


In [44]:
# Inspectăm rezultatele modelului într-un format mai ușor de citit

for i, row in results_df.iterrows():
    print("=" * 100)
    print(f"COMMENT {i + 1}")
    print("Source channel:", row["source_channel"])
    print("Video title:", row["video_title"])
    
    print("\nComment text:")
    print(row["comment_text"])
    
    print("\nModel output:")
    print(row["model_output"])
    print()

COMMENT 1
Source channel: georgesimionoficial
Video title: Turul României: realități de la firul ierbii

Comment text:
Felicitării George Simion Președintele României!🇷🇴😇⛑️🇷🇴❤️🛐❤️✝️✝️✝️❤️🕯️💐🇷🇴🙌💯🙌.

Model output:
```json
{
  "target": "George Simion",
  "stance": "supportive",
  "sentiment": "positive",
  "topic": "political leadership",
  "anti_establishment_frame": "no",
  "cynicism": "none",
  "interpretation_problem": "none"
}
```

COMMENT 2
Source channel: georgesimionoficial
Video title: Turul României: realități de la firul ierbii

Comment text:
Asa trebuie să fiți printre oameni nu sa se doarmă in parlament succes domnule Simion nu am greșit cind v-am votat

Model output:
```json
{
  "target": "George Simion (main), Parliament (secondary)",
  "stance": "supportive",
  "sentiment": "positive",
  "topic": "political leadership, parliamentary activity",
  "anti_establishment_frame": "no",
  "cynicism": "low",
  "interpretation_problem": "none"
}
```

COMMENT 3
Source channel: georg

Pentru comentariul : Felicitării George Simion Președintele României!🇷🇴😇⛑️🇷🇴❤️🛐❤️✝️✝️✝️❤️🕯️💐🇷🇴🙌💯🙌, targt recunoscut bine, stance este reperat bine - suportiv, putem vedea si emoticoanele care afirma acest aspect. sentimentul este pozitiv, mai acurat ar fi entuziasm pozitiv. Nu exista cinism, modelul a recunoscut si acest aspect ok.